In [1]:
import pandas as pd
import pickle as pkl
import numpy as np
import random
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn import metrics
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator
from custom_models.custom_model import BatchVectorizerClassifier
from ray import tune
from sklearn.utils.multiclass import unique_labels
from sklearn.linear_model import SGDOneClassSVM

In [2]:
neg_to_pos_ratio = 1
seed = 42
batch_size = 100000
ptype = 'is_guideline'
model_type = 'log'

In [3]:
filename_pos_human = '../data/pubmed_articles_pos_human.csv'

pos_data_human = pd.read_csv(
         filename_pos_human,
            chunksize=100000
)

In [4]:
filename_neg_human = '../data/pubmed_articles_neg_human.csv'

neg_data_human = pd.read_csv(filename_neg_human, chunksize=100000)

In [5]:
with open('../data/cleaned_pos_neg_train_val_test_dataset_ids.pkl', 'rb') as f:
    prop_typ_id_dict= pkl.load(f)

In [32]:
prop_typ_id_dict.keys()

dict_keys(['is_rct', 'is_sr', 'is_casecontrol', 'is_ct', 'is_cohort', 'is_guideline', 'is_consensus', 'is_qual', 'is_ctprotocol'])

In [6]:
pos_train_ids = prop_typ_id_dict[ptype]['pos']['train']
neg_train_ids = prop_typ_id_dict[ptype]['neg']['train']

In [7]:
pos_val_ids = prop_typ_id_dict[ptype]['pos']['val']
neg_val_ids = prop_typ_id_dict[ptype]['neg']['val']

In [8]:
pos_test_ids = prop_typ_id_dict[ptype]['pos']['test']
neg_test_ids = prop_typ_id_dict[ptype]['neg']['test']

In [9]:
pos_train_set = pd.DataFrame()
pos_val_set = pd.DataFrame()
pos_test_set = pd.DataFrame()

for pos_chunk in tqdm(pos_data_human):

    indices = pos_chunk.index
    indices = list(indices)
    
    
    pos_train_indices = list(set(indices).intersection(set(pos_train_ids)))
    pos_val_indices = list(set(indices).intersection(set(pos_val_ids)))
    pos_test_indices = list(set(indices).intersection(set(pos_test_ids)))
    
    pos_train_subset  = pos_chunk.loc[pos_train_indices]
    if pos_train_set.empty:
        pos_train_set = pd.DataFrame(columns = pos_train_set.columns)
    pos_train_set = pd.concat([pos_train_set, pos_train_subset])
    
    pos_val_subset  = pos_chunk.loc[pos_val_indices]
    if pos_val_set.empty:
        pos_val_set = pd.DataFrame(columns = pos_val_set.columns)
    pos_val_set = pd.concat([pos_val_set, pos_val_subset])
    
    pos_test_subset  = pos_chunk.loc[pos_test_indices]
    if pos_test_set.empty:
        pos_test_set = pd.DataFrame(columns = pos_test_set.columns)
    pos_test_set = pd.concat([pos_test_set, pos_test_subset])

pos_train_set[ptype] = int(1)
pos_val_set[ptype] = int(1)
pos_test_set[ptype] = int(1)

19it [00:29,  1.57s/it]/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/tqdm/std.py:1195: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:
51it [01:18,  1.53s/it]/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/tqdm/std.py:1195: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:
86it [02:11,  1.53s/it]


In [10]:
del pos_data_human
del pos_train_subset 
del pos_chunk
del indices
del pos_train_ids

In [11]:
neg_train_ids_sample = random.sample(neg_train_ids, round(len(pos_train_set)*neg_to_pos_ratio))

In [12]:
neg_train_set = pd.DataFrame()
neg_val_set = pd.DataFrame()
neg_test_set = pd.DataFrame()

for neg_chunk in tqdm(neg_data_human):

    indices = neg_chunk.index
    indices = list(indices)
    neg_train_indices = list(set(indices).intersection(set(neg_train_ids_sample)))
    neg_val_indices = list(set(indices).intersection(set(neg_val_ids)))
    neg_test_indices = list(set(indices).intersection(set(neg_test_ids)))
    neg_subset  = neg_chunk.loc[indices]
    
    
    neg_train_subset  = neg_chunk.loc[neg_train_indices]
    if neg_train_set.empty:
        neg_train_set = pd.DataFrame(columns = neg_train_set.columns)
    neg_train_set = pd.concat([neg_train_set, neg_train_subset])
    
    neg_val_subset  = neg_chunk.loc[neg_val_indices]
    if neg_val_set.empty:
        neg_val_set = pd.DataFrame(columns = neg_val_set.columns)
    neg_val_set = pd.concat([neg_val_set, neg_val_subset])
    
    neg_test_subset  = neg_chunk.loc[neg_test_indices]
    if neg_test_set.empty:
        neg_test_set = pd.DataFrame(columns = neg_test_set.columns)
    neg_test_set = pd.concat([neg_test_set, neg_test_subset])

neg_train_set['is_guideline'] = int(0)
neg_val_set['is_guideline'] = int(0)
neg_test_set['is_guideline'] = int(0)

57it [01:43,  1.82s/it]


In [13]:
train_data = pd.concat([neg_train_set, pos_train_set]).sample(frac=1)

In [14]:
del neg_train_set
del pos_train_set

In [15]:
val_data = pd.concat([neg_val_set, pos_val_set]).sample(frac=1)

In [16]:
del neg_val_set
del pos_val_set

In [17]:
test_data = pd.concat([neg_test_set, pos_test_set]).sample(frac=1)

In [18]:
del neg_test_set
del pos_test_set

In [19]:
X_train = pd.Series(["{0} {1}".format(train_title, train_abstract) for train_title, train_abstract in zip(train_data['title'], train_data['abstract_plaintext'])])
X_val = pd.Series(["{0} {1}".format(val_title, val_abstract) for val_title, val_abstract in zip(val_data['title'], val_data['abstract_plaintext'])])
X_test = pd.Series(["{0} {1}".format(test_title, test_abstract) for test_title, test_abstract in zip(test_data['title'], test_data['abstract_plaintext'])])


In [20]:
y_train = train_data[ptype]
y_val = val_data[ptype]
y_test = test_data[ptype]

In [21]:
del train_data
del val_data
del test_data

In [22]:
results = pd.read_json('results_linear_loss.json')

In [23]:
config = results[(results['ptype']==ptype) & (results['model_type'] == model_type)]

In [24]:
bvc = BatchVectorizerClassifier(loss=model_type, penalty=config['best_penalty'][0], alpha=config['best_alpha'][0], l1_ratio=0.15, fit_intercept=True, max_iter=1000, tol=0.001, shuffle=True, verbose=0, epsilon=0.1, n_jobs=None, random_state=None, learning_rate='optimal', eta0=0.0, power_t=0.5, early_stopping=False, validation_fraction=0.1, n_iter_no_change=5, class_weight=None, warm_start=False, average=False)
bvc.fit(X_train, y_train)
y_pred_test = bvc.predict(X_test)

In [25]:
y_test_list = list(y_test)

In [60]:
false_negatives =[]
tp=0
fn =0
for i in range(len(y_pred_test)):
    if y_pred_test[i] ==1 and y_test_list[i] ==1:
        tp +=1
    if y_pred_test[i] ==1 and y_test_list[i] ==0:
        false_negatives.append(X_test[i])
        fn +=1

In [61]:
false_negatives[0]

"Treatment of IgA nephropathy and Henoch-Schönlein nephritis. Treatment options for primary IgA nephropathy (IgAN) and Henoch-Schönlein nephritis are still largely based on opinion or weak evidence. Consequently, the recent KDIGO Clinical Practice Guidelines for Glomerulonephritis have assigned low levels of evidence for almost all recommendations and suggestions related to these two diseases. In this Review, we describe an algorithm for structuring the treatment of IgAN depending on the clinical scenario. Key to therapeutic decision making is assessment of the individual's prognosis. Clinical parameters (such as proteinuria, hypertension, and impaired glomerular filtration rate [GFR]) are used to estimate risk, but the clinical value of the novel histological Oxford-MEST classification remains to be determined. If these parameters indicate a risk of progressive GFR loss, comprehensive supportive care remains the mainstay of therapy. Two large trials, STOP-IgAN and TESTING, are underwa

In [62]:
false_negatives[5]

'[Nutritional impact of radiotherapy in oncogeriatric patients. Can radiation oncologists minimize the radiotherapy impact on the nutritional status of the elderly patient?] Introduction\n\n\nCancer in elderly patients represents a global health challenge due to the increase in its incidence and its associated mortality. In our country it is the second leading cause of death in people over 65 years. In the coming years, the problem will increase, with a prognosis of people over 65 years of 35% in 2050. Radiotherapy is an essential part of the multidisciplinary treatment of cancer in elderly patients, where sometimes it is considered as the first therapeutic option and others, as an alternative to surgery and/or chemotherapy if they pose too much risk. The technological development of Radiation Oncology in recent years has allowed optimizing treatments and reducing side effects in elderly patients, who tend to have a higher incidence of comorbidities. It is essential that professionals 

In [63]:
len(false_negatives)

9137

In [64]:
len(y_test)

301195

In [66]:
metrics.recall_score(y_test, y_pred_test)

0.9671532846715328

In [67]:
metrics.precision_score(y_test, y_pred_test)

0.12664882431657426